# aug29 — the leak-free run (Kaggle T4 x2): GRPO with shuffled prompts

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN` (required),
`WANDB_API_KEY` (recommended; falls back to offline logging).

**The experiment every reader asks for**: every training run in the paper consumed
answer-ordered prompts. This notebook reruns the exact reported 7B recipe — same SFT warm
start, hypers, seed 0, `scale_rewards` default — with the one-line fix applied and
**verified behaviorally before any GPU time**: training prompts shuffled by the same
puzzle-seeded RNG as SFT data and every eval. Then: checkpoint curve on val, one-session
test eval (base / SFT / shuffled-peak / shuffled-final), and the copy-rule test on the
new endpoint.

**How to read the outcome** (written before running):

| Result | Reading |
|---|---|
| shuffled-final ≪ base on test, copy rate low, entropy-style collapse | the collapse is a real RLVR phenomenon, not the leak — the paper's original thesis returns, now leak-free |
| shuffled-final ≈ SFT or above base; no collapse | the leak was the cause; the paper is a clean post-mortem and says so |
| collapse but slower/shallower; or new degenerate behavior | intermediate — report the trajectory as measured; the copy-rule test on the new endpoint says what it converged to |
| training reward again saturates early with zero variance | check the read-back cell's output first — if prompts verified shuffled, this is a real result, not the bug |

Budget ~9 h against the 12 h cap: train ~5.5 h (single T4, original recipe; resumes from
the Hub checkpoint repo if the session dies) + serving/evals ~2.5 h. Run All resumes safely.


In [ ]:
# Cell 1 — setup. No `huggingface-cli login` (interactive prompt hang); HF_TOKEN env suffices.
import os, glob, json, subprocess, time
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
try: os.environ['WANDB_API_KEY'] = S.get_secret('WANDB_API_KEY')
except Exception: os.environ['WANDB_MODE'] = 'offline'; print('no WANDB secret -> offline')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
# trl PINNED to 1.10.0 -- the version the reported runs used (the paper's S6 verifies
# its scale_rewards default). Unpinned trl 1.12.0 fails at import against vllm 0.28.0
# (NCCLTrainerSendWeightsArgs), so vllm is NOT installed here: training never needs it,
# and Cell 4 installs it after training, for serving only.
!pip uninstall -y -q vllm 2>/dev/null || true
!pip install -q -e . openai peft trl==1.10.0 bitsandbytes accelerate
!pip show trl | grep -E '^(Name|Version)'
r = subprocess.run(['python','-c',
    'from trl import GRPOConfig, GRPOTrainer; import trl; print("trl", trl.__version__, "trainer import OK")'],
    capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'GRPOTrainer preflight import FAILED -- fix before training'
r = subprocess.run(['python','-c','import connections_rl; print("import OK")'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'not importable in subprocess'
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
r = subprocess.run(['python','-c',(
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert int(r.stdout.strip()) == 162, 'test split is not 162'
print('setup OK')


In [ ]:
# Cell 2 — THE FIX, plus behavioral read-back. Trains nothing until the dataset itself
# proves the prompts are shuffled (the lesson: verify the constructed object, not the source).
import re, random
p = 'src/connections_rl/train/grpo.py'
src = open(p).read()
anchor = 'words = [w.upper() for g in rec["answers"] for w in g["members"]]'
fixline = '        random.Random(int(rec.get("puzzle_id", rec.get("id", -1)))).shuffle(words)'
assert anchor in src, 'build_dataset anchor line not found -- grpo.py changed; stop and inspect'
if fixline.strip() not in src:
    src = src.replace(anchor, anchor + '\n' + fixline)
    if re.search(r'^import random$', src, re.M) is None:
        src = src.replace('import json', 'import json\nimport random', 1)
    open(p, 'w').write(src)
    print('patch applied in-session. Commit the same change to the repo (GRPO_PROMPT_FIX.md).')
else:
    print('fix already present in branch copy')

# READ-BACK: build the dataset and verify order is shuffled, deterministic, and complete.
check = r'''
from connections_rl.train.grpo import build_dataset, load_puzzle_split
recs = load_puzzle_split('data/splits', 'train')[:5]
ds1, ds2 = build_dataset(recs), build_dataset(recs)
for i, rec in enumerate(recs):
    answer_order = [w.upper() for g in rec['answers'] for w in g['members']]
    def words_of(ds):
        u = ds[i]['prompt'][1]['content']
        return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
    w1, w2 = words_of(ds1), words_of(ds2)
    assert sorted(w1) == sorted(answer_order), 'word multiset changed'
    assert w1 == w2, 'shuffle not deterministic'
    assert w1 != answer_order, 'PROMPT STILL IN ANSWER ORDER -- DO NOT TRAIN'
print('read-back OK: shuffled, deterministic, complete on', len(recs), 'records')
'''
r = subprocess.run(['python','-c',check], capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0, 'READ-BACK FAILED -- do not train'

# Config: the reported recipe, renamed end to end.
import yaml
cfg = yaml.safe_load(open('configs/train/grpo-7b.yaml'))
def rename(v):
    return v.replace('grpo-7b', 'grpo-7b-shuffled') if isinstance(v, str) else v
cfg = {k: rename(v) for k, v in cfg.items()}
os.makedirs('results-analysis/aug29', exist_ok=True)
yaml.safe_dump(cfg, open('results-analysis/aug29/grpo-7b-shuffled.yaml', 'w'))
print({k: v for k, v in cfg.items() if isinstance(v, str)})


In [ ]:
# Cell 3 — train (~5.5 h, single T4, original recipe; resumes from its Hub ckpt repo).
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
r = subprocess.run(['python','-m','connections_rl.train.grpo',
                    '--config','results-analysis/aug29/grpo-7b-shuffled.yaml'], env=env)
print('training exit:', r.returncode)
ckpts = sorted(glob.glob('artifacts/grpo-7b-shuffled/checkpoint-*'),
               key=lambda x: int(x.rsplit('-',1)[1]))
print('checkpoints:', [c.rsplit('-',1)[1] for c in ckpts])
assert ckpts, 'no checkpoints -- one retry maximum, then report and stop'


In [ ]:
# Cell 4 — one vLLM session for everything. vllm is installed only now, after
# training, so its version can never break the trainer import (see Cell 1).
!pip install -q vllm
!pip show vllm | grep -E '^(Name|Version)'
import urllib.request
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='adapters/sft-7b', token=os.environ['HF_TOKEN'])
ckpts = sorted(glob.glob('artifacts/grpo-7b-shuffled/checkpoint-*'), key=lambda x: int(x.rsplit('-',1)[1]))
mods = ['connections-rl-sft-7b=adapters/sft-7b',
        'shuffled-final=artifacts/grpo-7b-shuffled']
mods += [f"shuffled-ckpt-{c.rsplit('-',1)[1]}={c}" for c in ckpts]
proc = subprocess.Popen(
    'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
    '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
    shell=True, stdout=open('/kaggle/working/vllm.log','w'), stderr=subprocess.STDOUT)
for _ in range(150):
    try: urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); break
    except Exception: time.sleep(10)
else: raise RuntimeError('vLLM failed -- see /kaggle/working/vllm.log')


In [ ]:
# Cell 5 — checkpoint curve on val -> pick peak -> one-session test eval, 4 arms.
arms_arg = ','.join(f"shuffled-ckpt-{c.rsplit('-',1)[1]}:{c.rsplit('-',1)[1]}" for c in ckpts)
r = subprocess.run(['python','-m','connections_rl.eval.checkpoint_curve',
                    '--arms', arms_arg,
                    '--puzzles', 'data/splits/puzzles_val.json',
                    '--out', 'results-analysis/aug29/ckpt-curve-7b-shuffled'])
assert r.returncode == 0, 'checkpoint curve failed'
pts = json.load(open('results-analysis/aug29/ckpt-curve-7b-shuffled.json'))
peak = max(pts, key=lambda p: p['semantic_groups_correct'])
PEAK = peak['step']
print('val-semantic peak: step', PEAK, '| full curve:',
      [(p['step'], round(p['semantic_groups_correct'], 4)) for p in pts])

ARMS = {'base': 'Qwen/Qwen2.5-7B-Instruct', 'sft': 'connections-rl-sft-7b',
        'shuffled-peak': f'shuffled-ckpt-{PEAK}', 'shuffled-final': 'shuffled-final'}
lines = ['puzzles: data/splits/puzzles_test.json',
         'out_dir: results-analysis/aug29/leakfree-session-test',
         'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
for a, m in ARMS.items():
    lines += [f'  - name: {a}', f'    model: {m}', '    temperature: 0.0']
open('results-analysis/aug29/leakfree_test.yaml','w').write('\n'.join(lines) + '\n')
r = subprocess.run(['python','-m','connections_rl.eval.run',
                    '--config','results-analysis/aug29/leakfree_test.yaml'])
assert r.returncode == 0


In [ ]:
# Cell 6 — copy-rule test on the new arms + verdict against the prespecified table.
import re as _re, random as _rnd
def parse_groups(text):
    m = _re.search(r'<ANSWER>(.*?)</ANSWER>', text, _re.S)
    if not m: return None
    gs = [[w.strip().upper() for w in gm.group(1).split(',')]
          for line in m.group(1).strip().splitlines()
          if (gm := _re.match(r'\s*Group \d+:\s*(.+)', line))]
    return gs if len(gs) == 4 else None
def prompt_words(pf):
    chat = json.loads(pf) if isinstance(pf, str) else pf
    u = next(m['content'] for m in chat if m['role'] == 'user')
    return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
summary = {'session': 'aug29 leakfree session', 'peak_step': PEAK, 'arms': {}}
for arm in ARMS:
    d = f'results-analysis/aug29/leakfree-session-test/{arm}'
    m = json.load(open(f'{d}/metrics.json')); s = m['summary']['OVERALL']
    parsed = quad = tot = pure = 0
    for line in open(f'{d}/generations.jsonl'):
        r = json.loads(line)
        w = prompt_words(r['prompt']); gs = parse_groups(r['generation'])
        if gs is None or len(w) != 16: continue
        parsed += 1
        quads = [set(w[i:i+4]) for i in range(0, 16, 4)]
        h = sum(1 for g in gs if set(g) in quads)
        quad += h; tot += 4; pure += (h == 4)
    summary['arms'][arm] = {
        'groups_mean_ci': s['groups_correct'], 'slots': round(s['groups_correct'][0]*162),
        'invalid_ci': s['invalid_rate'], 'reward_ci': s['reward'],
        'copy_group_rate': quad/tot if tot else None, 'pure_copy_rate': pure/parsed if parsed else None}
    print(f"{arm:<15} groups {s['groups_correct'][0]:.4f} "
          f"[{s['groups_correct'][1]:.3f},{s['groups_correct'][2]:.3f}] "
          f"slots {round(s['groups_correct'][0]*162)}/648  inv {s['invalid_rate'][0]:.3f}  "
          f"reward {s['reward'][0]:.4f}  copy {quad/max(tot,1):.3f}/{pure/max(parsed,1):.3f}")
json.dump(summary, open('results-analysis/aug29/leakfree_summary.json','w'), indent=1)
print()
print('Anchors for comparison (prior sessions, labeled): base 26/648, SFT 52/648,')
print('leaked step-50 77-80/648, leaked final 4/648 (copy rate 92.4%).')
print('Read the outcome against the table in the header cell.')


In [ ]:
# Cell 7 — persist.
!zip -qr /kaggle/working/aug29-leakfree-outputs.zip results-analysis/aug29 artifacts/grpo-7b-shuffled/checkpoint-*/adapter_model.safetensors 2>/dev/null || zip -qr /kaggle/working/aug29-leakfree-outputs.zip results-analysis/aug29
print('zip ready: /kaggle/working/aug29-leakfree-outputs.zip')
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(
        folder_path='results-analysis/aug29',
        repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
        path_in_repo='aug29')
    print('Hub upload OK -> connections-rl-results/aug29')
except Exception as e:
    print('Hub upload skipped/failed (fine -- use the zip):', e)
